# 004 · Roots of Polynomials

| | |
|---|---|
| **Track** | Python for Calculus |
| **Level** | 1 — Objects of calculus |
| **Prerequisites** | 002 and 003 |
| **Next** | 005 · Graphs of Functions |

**Place in the path.** A root of \(P\) is a number \(r\) such that \(P(r) = 0\).
Lesson 002 found two roots of \(x^2 - 5x + 6\) by guessing. This lesson solves polynomials without guessing,
and it checks every root list by Vieta, not by trusting a printout.

## Learning objectives

By the end of this lesson you can:

- distinguish a root, a factor, and a multiplicity
- solve a polynomial exactly with SymPy and numerically with NumPy
- check a root list against Vieta's formulas
- explain why a real polynomial's non-real roots arrive in conjugate pairs
- say why `numpy.roots` and `sympy.solve` need not print roots in the same order


## 1. What a root is

**See.** The graph of \(y = P(x)\) crosses or touches the horizontal axis at a real root.
A complex root does not appear on that real graph. It is still a root.

**Name.** SymPy is the exact-mathematics library used for the rest of this course.
`sp.symbols("x")` creates a mathematical unknown, not a Python float.
`sp.solve(equation, x)` returns a list of solutions.
`sp.factor` writes a polynomial as a product of lower-degree pieces.
`sp.roots` returns a dictionary mapping each root to its multiplicity.

Install the scientific stack once, from the folder that contains `requirements.txt`:

```text
py -m pip install -r requirements.txt
```


In [ ]:
import sympy as sp

x = sp.symbols("x")
polynomial = x ** 2 - 5 * x + 6
print("factored:", sp.factor(polynomial))
print("roots:", sp.solve(polynomial, x))
print("multiplicities:", sp.roots(polynomial))

assert sp.solve(polynomial, x) == [2, 3]
assert sp.roots(polynomial) == {2: 1, 3: 1}


**Read the output.** \(x^2 - 5x + 6 = (x-2)(x-3)\). The roots are 2 and 3, each of multiplicity one.
This is the polynomial `polynomial(x)` from lesson 002. The guesses there are now a solved list.

**Check a second way — Vieta.** For \(x^2 + bx + c\) wait: write the monic quadratic as \(x^2 - (\text{sum})x + (\text{product})\).
For \(x^2 - 5x + 6\), the roots add to 5 and multiply to 6.


In [ ]:
import sympy as sp

x = sp.symbols("x")
roots = sp.solve(x ** 2 - 5 * x + 6, x)
assert sum(roots) == 5
assert sp.prod(roots) == 6
print("Vieta holds: sum", sum(roots), "product", sp.prod(roots))


## 2. A quadratic with rational roots

**Do.** Solve \(2x^2 - 7x + 3\). The leading coefficient is not 1, so clear it carefully.
Vieta for \(ax^2 + bx + c\) says the roots add to \(-b/a\) and multiply to \(c/a\).


In [ ]:
import sympy as sp

x = sp.symbols("x")
polynomial = 2 * x ** 2 - 7 * x + 3
roots = sp.solve(polynomial, x)
print(sp.factor(polynomial))
print(roots)

a, b, c = 2, -7, 3
assert set(roots) == {sp.Rational(1, 2), 3}
assert sum(roots) == sp.Rational(-b, a)
assert sp.prod(roots) == sp.Rational(c, a)
print("Vieta holds")


**Read the output.** SymPy returned `1/2`, a rational number, not the float `0.5`.
That is the float lesson from 001, repaired. `0.5` would be legal and slightly dirty.
`sp.Rational(1, 2)` is exact.


## 3. Multiplicity

**See.** \((x-1)^2\) touches the axis at 1 and does not cross it.
There is one root, counted twice. Calling `solve` hides the count. Calling `roots` keeps it.

**Misconception.** Two copies of the same number in a hand calculation mean the solver is broken.

**Correction.** A repeated root is real information. The derivative also vanishes there.
Lesson 010 will call that a critical point, and it is why multiplicity matters.


In [ ]:
import sympy as sp

x = sp.symbols("x")
polynomial = (x - 1) ** 2
print("solve, which lists 1 once:", sp.solve(polynomial, x))
print("roots, which counts:", sp.roots(polynomial))
assert sp.roots(polynomial) == {1: 2}

# Derivative test: a double root of P is a root of P'.
assert sp.simplify(sp.diff(polynomial, x).subs(x, 1)) == 0
print("P and P' both vanish at the double root")


## 4. A cubic, checked two ways

**Do.** Solve \(2x^3 - 3x^2 - 11x + 6\).

SymPy gives exact roots. NumPy's `numpy.roots` gives numerical approximations from the coefficient list.
The coefficient list is highest degree first: `[2, -3, -11, 6]`. Reversing it is the usual bug.

The two lists need not be printed in the same order. Compare them as sets of numbers, within a tolerance.


In [ ]:
import numpy as np
import sympy as sp

x = sp.symbols("x")
polynomial = 2 * x ** 3 - 3 * x ** 2 - 11 * x + 6
exact = sp.solve(polynomial, x)
print("factored:", sp.factor(polynomial))
print("exact:", exact)

coefficients = [2, -3, -11, 6]
numeric = np.roots(coefficients)
print("numeric:", numeric)

exact_as_complex = sorted((complex(value.evalf()) for value in exact), key=lambda z: (z.real, z.imag))
numeric_sorted = sorted(numeric, key=lambda z: (z.real, z.imag))
for left, right in zip(exact_as_complex, numeric_sorted):
    assert abs(left - right) < 1e-8

# Vieta for ax^3 + bx^2 + cx + d: sum of roots = -b/a
assert sum(exact) == sp.Rational(3, 2)
print("exact and numeric roots agree, and the sum matches Vieta")


**Read the output.** The exact roots are \(-2\), \(\tfrac{1}{2}\), and \(3\).
Their sum is \(\tfrac{3}{2}\), which is \(-b/a = 3/2\). NumPy prints floats and a different order.
Both descriptions are the same three numbers.


## 5. When the roots leave the real line

**See.** Lesson 002 classified \(x^2 + 4x + 5\) as a complex-conjugate pair, because its discriminant is \(16 - 20 = -4\).

**Do.** Solve it. Check three facts:

1. The roots are \(-2+i\) and \(-2-i\).
2. They are conjugates of each other. Real coefficients force this.
3. Vieta: they add to \(-4\) and multiply to \(5\).


In [ ]:
import sympy as sp

x = sp.symbols("x")
polynomial = x ** 2 + 4 * x + 5
roots = sp.solve(polynomial, x)
print(roots)

assert set(roots) == {-2 + sp.I, -2 - sp.I}
assert sp.simplify(sp.conjugate(roots[0]) - roots[1]) == 0 or sp.simplify(sp.conjugate(roots[1]) - roots[0]) == 0
assert sum(roots) == -4
assert sp.expand(sp.prod(roots)) == 5
print("conjugate pair, and Vieta holds")


**Misconception.** NumPy "found different roots" because it printed `+0.j` or swapped the order.

**Correction.** Compare values inside a tolerance, after sorting. Do not compare printed text.


## Workshop — four polynomials, one table

For each polynomial, print the exact roots and the Vieta checks (sum and product, scaled by the leading coefficient).
The four polynomials are the worked set of this lesson:

- \(x^2 - 5x + 6\)
- \(2x^2 - 7x + 3\)
- \(2x^3 - 3x^2 - 11x + 6\)
- \(x^2 + 4x + 5\)


In [ ]:
import sympy as sp

x = sp.symbols("x")
problems = [
    x ** 2 - 5 * x + 6,
    2 * x ** 2 - 7 * x + 3,
    2 * x ** 3 - 3 * x ** 2 - 11 * x + 6,
    x ** 2 + 4 * x + 5,
]

for polynomial in problems:
    degree = sp.degree(polynomial)
    leading = sp.LC(polynomial)
    roots = sp.solve(polynomial, x)
    # For a monic polynomial, sum of roots = -coefficient of x**(n-1).
    # In general that sum is -coeff(x**(n-1)) / leading.
    coefficient = polynomial.coeff(x, degree - 1)
    expected_sum = sp.simplify(-coefficient / leading)
    print(polynomial, "->", roots, "| sum", sum(roots), "expected", expected_sum)
    assert sp.simplify(sum(roots) - expected_sum) == 0

print("workshop passed")


## Mistakes this lesson is meant to prevent

- Reading `numpy.roots([c0, c1, c2])` as constant-term first. NumPy wants the highest degree first.
- Using `solve` when you needed the multiplicity. Use `roots`.
- Testing float roots with `==`.
- Dropping a complex root because it is invisible on the real graph.
- Forgetting the leading coefficient in Vieta's sum: it is \(-b/a\), not \(-b\), when \(a \neq 1\).

## What you now have, and what comes next

You can find roots exactly and numerically, and you can audit them.
Lesson 005 draws the real graphs those roots live on.
A real root is an intercept. A repeated root is a touch. A complex root is a graph that misses the axis.
